In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

import seaborn as sns
import matplotlib.pyplot as plt

import gc
import os

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

### Custom functions

In [ ]:
def pickle_dump(path, saveobj):
    import pickle
    filehandler = open(path,"wb")
    pickle.dump(saveobj,filehandler)
    print("File pickled")
    filehandler.close()

In [ ]:
def pickle_load(path):
    import pickle
    file = open(path,'rb')
    loadobj = pickle.load(file)
    file.close()
    return loadobj

In [ ]:
sampleSubmit_df = pd.read_csv("/kaggle/input/ncaam-march-mania-2021/MSampleSubmissionStage1.csv")
season_df = pd.read_csv("/kaggle/input/ncaam-march-mania-2021/MRegularSeasonCompactResults.csv")
tourney_df = pd.read_csv("/kaggle/input/ncaam-march-mania-2021/MNCAATourneyCompactResults.csv")
seeds_df = pd.read_csv('../input/ncaam-march-mania-2021/MNCAATourneySeeds.csv')
ordinals_df = pd.read_csv('../input/ncaam-march-mania-2021/MMasseyOrdinals.csv')
teams_df = pd.read_csv('../input/ncaam-march-mania-2021/MTeams.csv')

### Season Results Compact

In [ ]:
print(season_df.shape)
print("----------------------")
print(season_df.head())
print("----------------------")
print(season_df['WLoc'].value_counts())
print("----------------------")
print("Max DayNum: {}".format(season_df['DayNum'].max()))

### Prepare dataset for feature engineering

In [ ]:
team1_df = tourney_df.copy()
team2_df = tourney_df.copy()

team1_df['Team'] = team1_df['WTeamID']
team1_df = team1_df.reset_index(drop=True)

team2_df['Team'] = team2_df['LTeamID']
team2_df = team2_df.reset_index(drop=True)

tourneyData_df = pd.concat([team1_df, team2_df], ignore_index=True)
tourneyData_df.drop_duplicates(inplace=True)
tourneyData_df = tourneyData_df.reset_index(drop=True)
tourneyData_df['gameType'] = 'NCAA'
tourneyData_df.shape

In [ ]:
team1_df = season_df.copy()
team2_df = season_df.copy()

team1_df['Team'] = team1_df['WTeamID']
team1_df = team1_df.reset_index(drop=True)

team2_df['Team'] = team2_df['LTeamID']
team2_df = team2_df.reset_index(drop=True)

seasonData_df = pd.concat([team1_df, team2_df], ignore_index=True)
seasonData_df.drop_duplicates(inplace=True)
seasonData_df = seasonData_df.reset_index(drop=True)
seasonData_df['gameType'] = 'Season'
seasonData_df.shape

In [ ]:
features_df = pd.concat([tourneyData_df, seasonData_df], ignore_index=True)
features_df.sort_values(by=['Team','Season','DayNum'], inplace=True)
features_df['Opp'] = np.where(features_df['Team'] != features_df['WTeamID'], features_df['WTeamID'], features_df['LTeamID'])
features_df = features_df.drop_duplicates().reset_index(drop=True)
features_df.shape

In [ ]:
features_df.iloc[180:190,:]

## Feature Engineering

### Wins, games and win ratios

In [ ]:
%%time

nwins = 0
nhomewins = 0
nawaywins = 0
neutralwins = 0
ngames = 0
nhomegames = 0
nawaygames = 0
neutralgames = 0

nwins_season = 0
nhomewins_season = 0
nawaywins_season = 0
neutralwins_season = 0
ngames_season = 0
nhomegames_season = 0
nawaygames_season = 0
neutralgames_season = 0

winRatio = 0
hWinRatio = 0
aWinRatio = 0
nWinRatio = 0

winRatio_season = 0
hWinRatio_season = 0
aWinRatio_season = 0
nWinRatio_season = 0

numWins = []
numHomeWins = []
numAwayWins = []
numNeutralWins = []
numGames = []
numHomeGames = []
numAwayGames = []
numNeutralGames = []

numWins_season = []
numHomeWins_season = []
numAwayWins_season = []
numNeutralWins_season = []
numGames_season = []
numHomeGames_season = []
numAwayGames_season = []
numNeutralGames_season = []

allWinRatio = []
homeWinRatio = []
awayWinRatio = []
neutralWinRatio = []

allWinRatio_season = []
homeWinRatio_season = []
awayWinRatio_season = []
neutralWinRatio_season = []

curTeam = 0
curSeason = 0

for row in features_df.itertuples():
    
    # Next Team
    if getattr(row,'Team') != curTeam:
        curTeam = getattr(row,'Team')
        curSeason = getattr(row,'Season')
        
        nwins = 0
        nhomewins = 0
        nawaywins = 0
        neutralwins = 0
        ngames = 0
        nhomegames = 0
        nawaygames = 0
        neutralgames = 0

        nwins_season = 0
        nhomewins_season = 0
        nawaywins_season = 0
        neutralwins_season = 0
        ngames_season = 0
        nhomegames_season = 0
        nawaygames_season = 0
        neutralgames_season = 0

        winRatio = 0
        hWinRatio = 0
        aWinRatio = 0
        nWinRatio = 0

        winRatio_season = 0
        hWinRatio_season = 0
        aWinRatio_season = 0
        nWinRatio_season = 0

    # Next Season
    elif getattr(row,'Season') != curSeason:
        curSeason = getattr(row,'Season')
        
        nwins_season = 0
        nhomewins_season = 0
        nawaywins_season = 0
        neutralwins_season = 0
        ngames_season = 0
        nhomegames_season = 0
        nawaygames_season = 0
        neutralgames_season = 0
        
        winRatio_season = 0
        hWinRatio_season = 0
        aWinRatio_season = 0
        nWinRatio_season = 0
    
    # Calculate feature value for row
    ngames+=1; ngames_season+=1;
    if getattr(row,'WLoc') == 'H': nhomegames+=1; nhomegames_season+=1;
    if getattr(row,'WLoc') == 'A': nawaygames+=1; nawaygames_season+=1;
    if getattr(row,'WLoc') == 'N': neutralgames+=1; neutralgames_season+=1;
    
    if getattr(row,'Team') == getattr(row,'WTeamID'): 
        nwins+=1; nwins_season+=1; 
        if getattr(row,'WLoc') == 'H': nhomewins+=1; nhomewins_season+=1;
        if getattr(row,'WLoc') == 'A': nawaywins+=1; nawaywins_season+=1;
        if getattr(row,'WLoc') == 'N': neutralwins+=1; neutralwins_season+=1;
    
    winRatio = 0 if ngames == 0 else nwins/ngames
    hWinRatio = 0 if nhomegames == 0 else nhomewins/nhomegames
    aWinRatio = 0 if nawaygames == 0 else nawaywins/nawaygames
    nWinRatio = 0 if neutralgames == 0 else neutralwins/neutralgames
    
    winRatio_season = 0 if ngames_season == 0 else nwins_season/ngames_season
    hWinRatio_season = 0 if nhomegames_season == 0 else nhomewins_season/nhomegames_season
    aWinRatio_season = 0 if nawaygames_season == 0 else nawaywins_season/nawaygames_season
    nWinRatio_season = 0 if neutralgames_season == 0 else neutralwins_season/neutralgames_season

    # Append feature values for row
    numWins.append(nwins)
    numHomeWins.append(nhomewins)
    numAwayWins.append(nawaywins)
    numNeutralWins.append(neutralwins)
    numGames.append(ngames)
    numHomeGames.append(nhomegames)
    numAwayGames.append(nawaygames)
    numNeutralGames.append(neutralgames)

    numWins_season.append(nwins_season)
    numHomeWins_season.append(nhomewins_season)
    numAwayWins_season.append(nawaywins_season)
    numNeutralWins_season.append(neutralwins_season)
    numGames_season.append(ngames_season)
    numHomeGames_season.append(nhomegames_season)
    numAwayGames_season.append(nawaygames_season)
    numNeutralGames_season.append(neutralgames_season)

    allWinRatio.append(winRatio)
    homeWinRatio.append(hWinRatio)
    awayWinRatio.append(aWinRatio)
    neutralWinRatio.append(nWinRatio)

    allWinRatio_season.append(winRatio_season)
    homeWinRatio_season.append(hWinRatio_season)
    awayWinRatio_season.append(aWinRatio_season)
    neutralWinRatio_season.append(nWinRatio_season)

In [ ]:
# Custom features
features_df['spy_numWins'] = numWins
features_df['spy_numHomeWins'] = numHomeWins
features_df['spy_numAwayWins'] = numAwayWins
features_df['spy_numNeutralWins'] = numNeutralWins
features_df['spy_numGames'] = numGames
features_df['spy_numHomeGames'] = numHomeGames
features_df['spy_numAwayGames'] = numAwayGames
features_df['spy_numNeutralGames'] = numNeutralGames

features_df['spy_numWins_season'] = numWins_season
features_df['spy_numHomeWins_season'] = numHomeWins_season
features_df['spy_numAwayWins_season'] = numAwayWins_season
features_df['spy_numNeutralWins_season'] = numNeutralWins_season
features_df['spy_numGames_season'] = numGames_season
features_df['spy_numHomeGames_season'] = numHomeGames_season
features_df['spy_numAwayGames_season'] = numAwayGames_season
features_df['spy_numNeutralGames_season'] = numNeutralGames_season

features_df['spy_allWinRatio'] = allWinRatio
features_df['spy_homeWinRatio'] = homeWinRatio
features_df['spy_awayWinRatio'] = awayWinRatio
features_df['spy_neutralWinRatio'] = neutralWinRatio

features_df['spy_allWinRatio_season'] = allWinRatio_season
features_df['spy_homeWinRatio_season'] = homeWinRatio_season
features_df['spy_awayWinRatio_season'] = awayWinRatio_season
features_df['spy_neutralWinRatio_season'] = neutralWinRatio_season

In [ ]:
features_df.shape

In [ ]:
features_df.head()

In [ ]:
# dropCols = [col for col in features_df.columns if 'spy_' in col and '_season' not in col]
# features_df.drop(columns=dropcols, axis=1, inplace=True)

### Team head to head record

In [ ]:
features_df.head()

In [ ]:
features_df['isGame'] = 1

features_df['isWin'] = np.where(features_df['Team'] == features_df['WTeamID'], 1, 0)

features_df['isHome'] = np.where((features_df['Team'] == features_df['WTeamID']) & (features_df['WLoc']=='H'), 1,
                                np.where((features_df['Team'] == features_df['LTeamID']) & (features_df['WLoc']=='A'), 1,
                                        np.where((features_df['WLoc']=='N'), 0, 0)))

features_df['isHomeWin'] = np.where((features_df['isWin']==1) & (features_df['isHome']==1), 1, 0)

features_df['TeamH2H'] = np.where(features_df['Team'] == features_df['WTeamID'], 
                                  features_df['Team'].astype(str) + '-' + features_df['LTeamID'].astype(str),
                                  features_df['Team'].astype(str) + '-' + features_df['WTeamID'].astype(str))

features_df.head()

In [ ]:
temp = features_df.groupby('TeamH2H').agg(
            spy_numGames_H2H = ('isGame',sum),
            spy_numWins_H2H = ('isWin',sum),
            spy_homeGames_H2H = ('isHome',sum),
            spy_homeWins_H2H = ('isHomeWin',sum),
        ).reset_index(drop=False)

temp['spy_winRatio_H2H'] = np.where(temp['spy_numGames_H2H']!=0, temp['spy_numWins_H2H'] / temp['spy_numGames_H2H'], 0)
temp['spy_homeWinRatio_H2H'] = np.where(temp['spy_homeGames_H2H']!=0, temp['spy_homeWins_H2H'] / temp['spy_homeGames_H2H'], 0)

temp.head()

In [ ]:
features_df = pd.merge(features_df, temp, on="TeamH2H", how="left")
features_df.shape

In [ ]:
features_df.head()

### Ordinal Ranking

In [ ]:
ordinals_df.shape

#### Last day of season rankings

In [ ]:
ordinals_df['gameType'] = np.where(ordinals_df['RankingDayNum'] < 134, 'Season', 'NCAA')
ordinals_df.head()

In [ ]:
# temp = ordinals_df[ordinals_df['gameType']=='Season'].copy()
# # temp1 = ordinals_df.groupby(['Season','TeamID','gameType'])['RankingDayNum'].max().reset_index(drop=False)
# temp1 = ordinals_df.groupby(['Season','TeamID','gameType','RankingDayNum']).reset_index(drop=False)
# temp2 = pd.merge(temp1, ordinals_df,on=['Season','TeamID','gameType','RankingDayNum'], how="left")
ordinal_feats_df = ordinals_df.groupby(['Season','TeamID','gameType','RankingDayNum']).agg(
    spy_maxOrdinalRank = ('OrdinalRank',max),
    spy_minOrdinalRank = ('OrdinalRank',min),
    spy_rangeOrdinalRank = ('OrdinalRank', lambda x: max(x)-min(x)),
    spy_avgOrdinalRank = ('OrdinalRank',lambda x: round(sum(x) / len(x)))
).reset_index(drop=False)
ordinal_feats_df.rename(columns={'TeamID':'Team'}, inplace=True)
ordinal_feats_df.drop(columns=['gameType'], inplace=True)
ordinal_feats_df.head()

In [ ]:
ordinals_df[(ordinals_df['TeamID']==1102) &
            (ordinals_df['Season']==2003) & 
            (ordinals_df['RankingDayNum']==35)]

In [ ]:
ordinal_feats_df.shape

### Seed ranking

In [ ]:
seeds_df['spy_SeedNum'] = (seeds_df['Seed'].str[1:3]).astype(int)
seed_feats_df = seeds_df.drop(columns=['Seed'])
seed_feats_df['spy_seeded'] = 1
seed_feats_df.rename(columns={'TeamID':'Team'}, inplace=True)
seed_feats_df['Season'] = seed_feats_df['Season']+1
seed_feats_df.head()

In [ ]:
features_df.shape

In [ ]:
features_df = pd.merge(features_df, seed_feats_df, on=['Season', 'Team'], how="left")

features_df['spy_SeedNum'] = features_df['spy_SeedNum'].fillna(20)
features_df['spy_seeded'] = features_df['spy_seeded'].fillna(0)
features_df.shape

In [ ]:
# features_df = pd.merge(features_df, ordinal_feats_df, on=['Season', 'Team'], how="left")
# features_df.shape

In [ ]:
features_df.dtypes

## Create modeling dataset

In [ ]:
model_df1 = tourney_df[tourney_df['Season'] >= 2005].copy()
model_df2 = tourney_df[tourney_df['Season'] >= 2005].copy()

model_df1 = model_df1.loc[:, ['Season','WTeamID','LTeamID']]
model_df1.columns = ['Season','TeamID1','TeamID2']
model_df1['result'] = 1

model_df2 = model_df2.loc[:, ['Season','LTeamID','WTeamID']]
model_df2.columns = ['Season','TeamID1','TeamID2']
model_df2['result'] = 0

tourney_model_df = pd.concat([model_df1, model_df2])

del model_df1, model_df2
gc.collect()

tourney_model_df.head(1)

In [ ]:
model_df1 = season_df[season_df['Season'] >= 2005].copy()
model_df2 = season_df[season_df['Season'] >= 2005].copy()

model_df1 = model_df1.loc[:, ['Season','WTeamID','LTeamID']]
model_df1.columns = ['Season','TeamID1','TeamID2']
model_df1['result'] = 1

model_df2 = model_df2.loc[:, ['Season','LTeamID','WTeamID']]
model_df2.columns = ['Season','TeamID1','TeamID2']
model_df2['result'] = 0

season_model_df = pd.concat([model_df1, model_df2])

del model_df1, model_df2
gc.collect()

season_model_df.head(1)

In [ ]:
model_df = pd.concat([tourney_model_df, season_model_df])
model_df.shape

In [ ]:
print(model_df.shape)

train_df = pd.DataFrame()

for season in model_df.Season.unique():
    temp_df1 = features_df[(features_df['Season']==season) & (features_df['gameType']=='Season')].copy()
    temp_df2 = pd.DataFrame(temp_df1.groupby(['Team'])['DayNum'].max()).reset_index()
    featCols = [col for col in features_df.columns if 'spy_' in col]
    temp_df1 = temp_df1[['Team','DayNum','Season']+featCols]

    temp_df = pd.merge(temp_df2, temp_df1, on=['Team','DayNum'], how="left")
    
    team1_feats_df = temp_df.drop(['DayNum'], axis=1)
    team1_feats_df = team1_feats_df.add_prefix("Team1_")
    team1_feats_df.rename(columns = {'Team1_Team':'TeamID1','Team1_Season':'Season'}, inplace=True)

    team2_feats_df = temp_df.drop(['DayNum'], axis=1)
    team2_feats_df = team2_feats_df.add_prefix("Team2_")
    team2_feats_df.rename(columns = {'Team2_Team':'TeamID2','Team2_Season':'Season'}, inplace=True)
    
    model_df_temp = model_df[model_df['Season']==season].copy()

    model_df_temp = pd.merge(model_df_temp, team1_feats_df, on=['TeamID1','Season'], how="left")
    model_df_temp = pd.merge(model_df_temp, team2_feats_df, on=['TeamID2','Season'], how="left")
    
    train_df = train_df.append(model_df_temp)
    
print(train_df.shape)

In [ ]:
train_df.sort_values(by=['Season'], ascending=True, inplace=True)
train_df = train_df.reset_index(drop=True)
train_df.head()

In [ ]:
# Stat comparison features
train_df['spy_seedNumDiff'] = train_df['Team1_spy_SeedNum'] - train_df['Team2_spy_SeedNum']
train_df['spy_seededDiff'] = train_df['Team1_spy_seeded'] - train_df['Team2_spy_seeded']

# train_df['spy_maxOrdRnkDiff'] = train_df['Team1_spy_maxOrdinalRank'] - train_df['Team1_spy_maxOrdinalRank']
# train_df['spy_minOrdRnkDiff'] = train_df['Team1_spy_minOrdinalRank'] - train_df['Team1_spy_minOrdinalRank']
# train_df['spy_rangeOrdRnkDiff'] = train_df['Team1_spy_rangeOrdinalRank'] - train_df['Team1_spy_rangeOrdinalRank']
# train_df['spy_avgOrdRnkDiff'] = train_df['Team1_spy_avgOrdinalRank'] - train_df['Team1_spy_avgOrdinalRank']

train_df['spy_gameExpRatio'] = train_df['Team1_spy_numGames'] / train_df['Team2_spy_numGames']
train_df['spy_gameExpRatio_season'] = train_df['Team1_spy_numGames_season'] / train_df['Team2_spy_numGames_season']

train_df['spy_WinRecordRatio'] = train_df['Team1_spy_numWins'] / train_df['Team2_spy_numWins']
train_df['spy_WinRecordRatio_season'] = train_df['Team1_spy_numWins_season'] / train_df['Team2_spy_numWins_season']

train_df['spy_WinRatioRatio'] = train_df['Team1_spy_allWinRatio'] / train_df['Team2_spy_allWinRatio']
train_df['spy_WinRatioRatio_season'] = train_df['Team1_spy_allWinRatio_season'] / train_df['Team2_spy_allWinRatio_season']

In [ ]:
train_df['spy_numWinsRatio_H2H'] = np.where(train_df['Team2_spy_numWins_H2H'] != 0, 
                                            train_df['Team1_spy_numWins_H2H'] / train_df['Team2_spy_numWins_H2H'], 0)
train_df['spy_numHomeGamesRatio_H2H'] = np.where(train_df['Team2_spy_homeGames_H2H'] != 0, 
                                            train_df['Team1_spy_homeGames_H2H'] / train_df['Team2_spy_homeGames_H2H'], 0)
train_df['spy_numHomeWinsRatio_H2H'] = np.where(train_df['Team2_spy_homeWins_H2H'] != 0, 
                                            train_df['Team1_spy_homeWins_H2H'] / train_df['Team2_spy_homeWins_H2H'], 0)
train_df['spy_WinRatioRatio_H2H'] = np.where(train_df['Team2_spy_winRatio_H2H'] != 0, 
                                            train_df['Team1_spy_winRatio_H2H'] / train_df['Team2_spy_winRatio_H2H'], 0)
train_df['spy_homeWinRatioRatio_H2H'] = np.where(train_df['Team2_spy_homeWinRatio_H2H'] != 0, 
                                            train_df['Team1_spy_homeWinRatio_H2H'] / train_df['Team2_spy_homeWinRatio_H2H'], 0)

In [ ]:
print(train_df.shape)
train_df = train_df.dropna()
print(train_df.shape)

In [ ]:
train_df1 = train_df[(train_df['Team1_spy_SeedNum']!=20) & (train_df['Team2_spy_SeedNum']!=20)].copy()
print(train_df1.shape)
train_df2 = train_df[(train_df['Team1_spy_SeedNum']==20) | (train_df['Team2_spy_SeedNum']==20)].copy()
train_df2 = train_df2.sample(5000)
print(train_df2.shape)

train_df = pd.concat([train_df1, train_df2]).drop_duplicates().reset_index(drop=True)
print(train_df.shape)

In [ ]:
train_df['Team1_spy_SeedNum'].value_counts()

### GroupTimeSeriesSplit

In [ ]:
from sklearn.model_selection._split import _BaseKFold, indexable, _num_samples
from sklearn.utils.validation import _deprecate_positional_args

# https://github.com/getgaurav2/scikit-learn/blob/d4a3af5cc9da3a76f0266932644b884c99724c57/sklearn/model_selection/_split.py#L2243
class GroupTimeSeriesSplit(_BaseKFold):
    """Time Series cross-validator variant with non-overlapping groups.
    Provides train/test indices to split time series data samples
    that are observed at fixed time intervals according to a
    third-party provided group.
    In each split, test indices must be higher than before, and thus shuffling
    in cross validator is inappropriate.
    This cross-validation object is a variation of :class:`KFold`.
    In the kth split, it returns first k folds as train set and the
    (k+1)th fold as test set.
    The same group will not appear in two different folds (the number of
    distinct groups has to be at least equal to the number of folds).
    Note that unlike standard cross-validation methods, successive
    training sets are supersets of those that come before them.
    Read more in the :ref:`User Guide <cross_validation>`.
    Parameters
    ----------
    n_splits : int, default=5
        Number of splits. Must be at least 2.
    max_train_size : int, default=None
        Maximum size for a single training set.
    Examples
    --------
    >>> import numpy as np
    >>> from sklearn.model_selection import GroupTimeSeriesSplit
    >>> groups = np.array(['a', 'a', 'a', 'a', 'a', 'a',\
                           'b', 'b', 'b', 'b', 'b',\
                           'c', 'c', 'c', 'c',\
                           'd', 'd', 'd'])
    >>> gtss = GroupTimeSeriesSplit(n_splits=3)
    >>> for train_idx, test_idx in gtss.split(groups, groups=groups):
    ...     print("TRAIN:", train_idx, "TEST:", test_idx)
    ...     print("TRAIN GROUP:", groups[train_idx],\
                  "TEST GROUP:", groups[test_idx])
    TRAIN: [0, 1, 2, 3, 4, 5] TEST: [6, 7, 8, 9, 10]
    TRAIN GROUP: ['a' 'a' 'a' 'a' 'a' 'a']\
    TEST GROUP: ['b' 'b' 'b' 'b' 'b']
    TRAIN: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10] TEST: [11, 12, 13, 14]
    TRAIN GROUP: ['a' 'a' 'a' 'a' 'a' 'a' 'b' 'b' 'b' 'b' 'b']\
    TEST GROUP: ['c' 'c' 'c' 'c']
    TRAIN: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]\
    TEST: [15, 16, 17]
    TRAIN GROUP: ['a' 'a' 'a' 'a' 'a' 'a' 'b' 'b' 'b' 'b' 'b' 'c' 'c' 'c' 'c']\
    TEST GROUP: ['d' 'd' 'd']
    """
    @_deprecate_positional_args
    def __init__(self,
                 n_splits=5,
                 *,
                 max_train_size=None
                 ):
        super().__init__(n_splits, shuffle=False, random_state=None)
        self.max_train_size = max_train_size

    def split(self, X, y=None, groups=None):
        """Generate indices to split data into training and test set.
        Parameters
        ----------
        X : array-like of shape (n_samples, n_features)
            Training data, where n_samples is the number of samples
            and n_features is the number of features.
        y : array-like of shape (n_samples,)
            Always ignored, exists for compatibility.
        groups : array-like of shape (n_samples,)
            Group labels for the samples used while splitting the dataset into
            train/test set.
        Yields
        ------
        train : ndarray
            The training set indices for that split.
        test : ndarray
            The testing set indices for that split.
        """
        if groups is None:
            raise ValueError(
                "The 'groups' parameter should not be None")
        X, y, groups = indexable(X, y, groups)
        n_samples = _num_samples(X)
        n_splits = self.n_splits
        n_folds = n_splits + 1
        group_dict = {}
        u, ind = np.unique(groups, return_index=True)
        unique_groups = u[np.argsort(ind)]
        n_samples = _num_samples(X)
        n_groups = _num_samples(unique_groups)
        for idx in np.arange(n_samples):
            if (groups[idx] in group_dict):
                group_dict[groups[idx]].append(idx)
            else:
                group_dict[groups[idx]] = [idx]
        if n_folds > n_groups:
            raise ValueError(
                ("Cannot have number of folds={0} greater than"
                 " the number of groups={1}").format(n_folds,
                                                     n_groups))
        group_test_size = n_groups // n_folds
        group_test_starts = range(n_groups - n_splits * group_test_size,
                                  n_groups, group_test_size)
        for group_test_start in group_test_starts:
            train_array = []
            test_array = []
            for train_group_idx in unique_groups[:group_test_start]:
                train_array_tmp = group_dict[train_group_idx]
                train_array = np.sort(np.unique(
                                      np.concatenate((train_array,
                                                      train_array_tmp)),
                                      axis=None), axis=None)
            train_end = train_array.size
            if self.max_train_size and self.max_train_size < train_end:
                train_array = train_array[train_end -
                                          self.max_train_size:train_end]
            for test_group_idx in unique_groups[group_test_start:
                                                group_test_start +
                                                group_test_size]:
                test_array_tmp = group_dict[test_group_idx]
                test_array = np.sort(np.unique(
                                              np.concatenate((test_array,
                                                              test_array_tmp)),
                                     axis=None), axis=None)
            yield [int(i) for i in train_array], [int(i) for i in test_array]

In [ ]:
for fold,(train_idx, valid_idx) in enumerate(GroupTimeSeriesSplit().split(train_df, groups=train_df['Season'])):
        print(fold)
        train_temp = train_df.loc[train_idx,:]
        print(train_temp.shape)
        print(train_temp.Season.unique())
        valid_temp = train_df.loc[valid_idx,:]
        print(valid_temp.shape)
        print(valid_temp.Season.unique())

### Train model

In [ ]:
from sklearn.metrics import log_loss
from sklearn.metrics import roc_auc_score

In [ ]:
# Model parameters
from xgboost import XGBClassifier

xgb_params = {
 'colsample_bytree': 0.8,
 'gamma': 20.0,
 'learning_rate': 0.05,
 'max_depth': 5,
 'min_child_weight': 30,
 'reg_alpha': 0.03,
 'reg_lambda': 20,
 'n_estimators': 10000,
 'random_state': 101,
 'scale_pos_weight': 2,
 'subsample': 0.8,
 'n_jobs': -1
}

xgb_model = XGBClassifier(**xgb_params)

In [ ]:
best_loss = np.inf

for fold,(train_idx, valid_idx) in enumerate(GroupTimeSeriesSplit().split(train_df, groups=train_df['Season'])):
    
#     if fold > 2:
    
    print(fold)

    train_temp = train_df.loc[train_idx,:]
    valid_temp = train_df.loc[valid_idx,:]

    X_train = train_temp[[col for col in train_df.columns if 'spy_' in col]]
    y_train = train_temp[['result']]

    print(X_train.shape)
    print(y_train.shape)

    X_valid = valid_temp[[col for col in train_df.columns if 'spy_' in col]]
    y_valid = valid_temp[['result']]

    print(X_valid.shape)
    print(y_valid.shape)



    # Train model
    xgb_model.fit(X_train, y_train['result'].values.ravel(), early_stopping_rounds=50, eval_metric='auc', eval_set=[(X_valid, y_valid['result'].values.ravel())], verbose=1)

    # Train performance
    ytrain_pred_prob_xgb = xgb_model.predict_proba(X_train)
    ytrain_pred_xgb = xgb_model.predict(X_train)
    print('XGB train roc-auc: {}'.format(roc_auc_score(y_train['result'].values.ravel(), ytrain_pred_prob_xgb[:,1])))
    print('XGB train logloss: {}'.format(log_loss(y_train['result'].values.ravel(), ytrain_pred_prob_xgb)))
#         print('XGB train accuracy: {}'.format(accuracy_score(y_train['fpd'].values.ravel(), ytrain_pred_xgb)))
#         print('XGB train Confusion matrix: \n {}'.format(confusion_matrix(y_train['fpd'].values.ravel(), ytrain_pred_xgb)))
#         print('XGB train Classification report: \n {}'.format(classification_report(y_train['fpd'].values.ravel(), ytrain_pred_xgb)))

    # Test performance
    ytest_pred_prob_xgb = xgb_model.predict_proba(X_valid)
    ytest_pred_xgb = xgb_model.predict(X_valid)
    print('XGB train roc-auc: {}'.format(roc_auc_score(y_valid['result'].values.ravel(), ytest_pred_prob_xgb[:,1])))
    print('XGB test logloss: {}'.format(log_loss(y_valid['result'].values.ravel(), ytest_pred_prob_xgb)))
#         print('XGB test accuracy: {}'.format(accuracy_score(y_test['fpd'].values.ravel(), ytest_pred_xgb)))
#         print('XGB test Confusion matrix: \n {}'.format(confusion_matrix(y_test['fpd'].values.ravel(), ytest_pred_xgb)))
#         print('XGB test Classification report: \n {}'.format(classification_report(y_test['fpd'].values.ravel(), ytest_pred_xgb)))

    fold_loss = log_loss(y_valid['result'].values.ravel(), ytest_pred_prob_xgb)

    if fold_loss < best_loss:
        print("Best loss: {}".format(fold_loss))
        best_loss = fold_loss
        pickle_dump('/kaggle/working/xgb_model.pkl', xgb_model)
        print("Saved model object")


In [ ]:
xgb_model = pickle_load('/kaggle/working/xgb_model.pkl')
log_loss(train_df['result'].values.ravel(), xgb_model.predict_proba(train_df[[col for col in train_df.columns if 'spy_' in col]]))

In [ ]:
xgb_feat_imps = pd.Series(xgb_model.feature_importances_, index=[col for col in train_df.columns if 'spy_' in col])
xgb_feat_imps.nlargest(25).plot(kind='barh', figsize=(20,10))
plt.show()

In [ ]:
keepCols = xgb_feat_imps[xgb_feat_imps>0.01].index.tolist()

In [ ]:
best_loss = np.inf

for fold,(train_idx, valid_idx) in enumerate(GroupTimeSeriesSplit().split(train_df, groups=train_df['Season'])):
    
#     if fold > 2:
    
    print(fold)

    train_temp = train_df.loc[train_idx,:]
    valid_temp = train_df.loc[valid_idx,:]

    X_train = train_temp[keepCols]
    y_train = train_temp[['result']]

    print(X_train.shape)
    print(y_train.shape)

    X_valid = valid_temp[keepCols]
    y_valid = valid_temp[['result']]

    print(X_valid.shape)
    print(y_valid.shape)



    # Train model
    xgb_model.fit(X_train, y_train['result'].values.ravel(), early_stopping_rounds=50, eval_metric='auc', eval_set=[(X_valid, y_valid['result'].values.ravel())], verbose=1)

    # Train performance
    ytrain_pred_prob_xgb = xgb_model.predict_proba(X_train)
    ytrain_pred_xgb = xgb_model.predict(X_train)
    print('XGB train roc-auc: {}'.format(roc_auc_score(y_train['result'].values.ravel(), ytrain_pred_prob_xgb[:,1])))
    print('XGB train logloss: {}'.format(log_loss(y_train['result'].values.ravel(), ytrain_pred_prob_xgb)))
#         print('XGB train accuracy: {}'.format(accuracy_score(y_train['fpd'].values.ravel(), ytrain_pred_xgb)))
#         print('XGB train Confusion matrix: \n {}'.format(confusion_matrix(y_train['fpd'].values.ravel(), ytrain_pred_xgb)))
#         print('XGB train Classification report: \n {}'.format(classification_report(y_train['fpd'].values.ravel(), ytrain_pred_xgb)))

    # Test performance
    ytest_pred_prob_xgb = xgb_model.predict_proba(X_valid)
    ytest_pred_xgb = xgb_model.predict(X_valid)
    print('XGB train roc-auc: {}'.format(roc_auc_score(y_valid['result'].values.ravel(), ytest_pred_prob_xgb[:,1])))
    print('XGB test logloss: {}'.format(log_loss(y_valid['result'].values.ravel(), ytest_pred_prob_xgb)))
#         print('XGB test accuracy: {}'.format(accuracy_score(y_test['fpd'].values.ravel(), ytest_pred_xgb)))
#         print('XGB test Confusion matrix: \n {}'.format(confusion_matrix(y_test['fpd'].values.ravel(), ytest_pred_xgb)))
#         print('XGB test Classification report: \n {}'.format(classification_report(y_test['fpd'].values.ravel(), ytest_pred_xgb)))

    fold_loss = log_loss(y_valid['result'].values.ravel(), ytest_pred_prob_xgb)

    if fold_loss < best_loss:
        print("Best loss: {}".format(fold_loss))
        best_loss = fold_loss
        pickle_dump('/kaggle/working/xgb_model.pkl', xgb_model)
        print("Saved model object")


In [ ]:
xgb_model = pickle_load('/kaggle/working/xgb_model.pkl')
log_loss(train_df['result'].values.ravel(), xgb_model.predict_proba(train_df[keepCols]))

In [ ]:
xgb_feat_imps = pd.Series(xgb_model.feature_importances_, index=keepCols)
xgb_feat_imps.nlargest(25).plot(kind='barh', figsize=(20,10))
plt.show()

### Score submission

In [ ]:
sampleSubmit_df = pd.read_csv("/kaggle/input/ncaam-march-mania-2021/MSampleSubmissionStage1.csv")
sampleSubmit_df.head()

In [ ]:
sampleSubmit_df['Year'] = sampleSubmit_df['ID'].apply(lambda x: x.split('_')[0])
sampleSubmit_df['TeamID1'] = sampleSubmit_df['ID'].apply(lambda x: x.split('_')[1])
sampleSubmit_df['TeamID2'] = sampleSubmit_df['ID'].apply(lambda x: x.split('_')[2])

print(sampleSubmit_df.shape)
print(sampleSubmit_df.head())

In [ ]:
sampleSubmit_df.Year.unique()

In [ ]:
print(sampleSubmit_df.shape)

submit_df = pd.DataFrame()

for season in sampleSubmit_df.Year.unique():
    print(season)
    temp_df1 = features_df[(features_df['Season']==int(season)) & (features_df['gameType']=='Season')].copy()
    temp_df2 = pd.DataFrame(temp_df1.groupby(['Team'])['DayNum'].max()).reset_index()
    featCols = [col for col in features_df.columns if 'spy_' in col]
    temp_df1 = temp_df1[['Team','DayNum','Season']+featCols]

    temp_df = pd.merge(temp_df2, temp_df1, on=['Team','DayNum'], how="left")
    
    team1_feats_df = temp_df.drop(['DayNum'], axis=1)
    team1_feats_df = team1_feats_df.add_prefix("Team1_")
    team1_feats_df.rename(columns = {'Team1_Team':'TeamID1','Team1_Season':'Year'}, inplace=True)

    team2_feats_df = temp_df.drop(['DayNum'], axis=1)
    team2_feats_df = team2_feats_df.add_prefix("Team2_")
    team2_feats_df.rename(columns = {'Team2_Team':'TeamID2','Team2_Season':'Year'}, inplace=True)
    
    model_df_temp = sampleSubmit_df[sampleSubmit_df['Year']==str(season)].copy()
    
    model_df_temp['Year'] = model_df_temp['Year'].astype(int)
    team1_feats_df['Year'] = team1_feats_df['Year'].astype(int)
    team2_feats_df['Year'] = team2_feats_df['Year'].astype(int)

    model_df_temp['TeamID1'] = model_df_temp['TeamID1'].astype(int)
    team1_feats_df['TeamID1'] = team1_feats_df['TeamID1'].astype(int)

    model_df_temp['TeamID2'] = model_df_temp['TeamID2'].astype(int)
    team2_feats_df['TeamID2'] = team2_feats_df['TeamID2'].astype(int)  

    model_df_temp = pd.merge(model_df_temp, team1_feats_df, on=['TeamID1','Year'], how="left")
    model_df_temp = pd.merge(model_df_temp, team2_feats_df, on=['TeamID2','Year'], how="left")

    if submit_df.shape[0] == 0:
        submit_df = model_df_temp.copy()
    else:
        submit_df = submit_df.append(model_df_temp)
    
print(submit_df.shape)

In [ ]:
# Stat comparison features
submit_df['spy_seedNumDiff'] = submit_df['Team1_spy_SeedNum'] - submit_df['Team2_spy_SeedNum']
submit_df['spy_seededDiff'] = submit_df['Team1_spy_seeded'] - submit_df['Team2_spy_seeded']

# submit_df['spy_maxOrdRnkDiff'] = submit_df['Team1_spy_maxOrdinalRank'] - submit_df['Team1_spy_maxOrdinalRank']
# submit_df['spy_minOrdRnkDiff'] = submit_df['Team1_spy_minOrdinalRank'] - submit_df['Team1_spy_minOrdinalRank']
# submit_df['spy_rangeOrdRnkDiff'] = submit_df['Team1_spy_rangeOrdinalRank'] - submit_df['Team1_spy_rangeOrdinalRank']
# submit_df['spy_avgOrdRnkDiff'] = submit_df['Team1_spy_avgOrdinalRank'] - submit_df['Team1_spy_avgOrdinalRank']

submit_df['spy_gameExpRatio'] = submit_df['Team1_spy_numGames'] / submit_df['Team2_spy_numGames']
submit_df['spy_gameExpRatio_season'] = submit_df['Team1_spy_numGames_season'] / submit_df['Team2_spy_numGames_season']

submit_df['spy_WinRecordRatio'] = submit_df['Team1_spy_numWins'] / submit_df['Team2_spy_numWins']
submit_df['spy_WinRecordRatio_season'] = submit_df['Team1_spy_numWins_season'] / submit_df['Team2_spy_numWins_season']

submit_df['spy_WinRatioRatio'] = submit_df['Team1_spy_allWinRatio'] / submit_df['Team2_spy_allWinRatio']
submit_df['spy_WinRatioRatio_season'] = submit_df['Team1_spy_allWinRatio_season'] / submit_df['Team2_spy_allWinRatio_season']

In [ ]:
submit_df['spy_numWinsRatio_H2H'] = np.where(submit_df['Team2_spy_numWins_H2H'] != 0, 
                                            submit_df['Team1_spy_numWins_H2H'] / submit_df['Team2_spy_numWins_H2H'], 0)
submit_df['spy_numHomeGamesRatio_H2H'] = np.where(submit_df['Team2_spy_homeGames_H2H'] != 0, 
                                            submit_df['Team1_spy_homeGames_H2H'] / submit_df['Team2_spy_homeGames_H2H'], 0)
submit_df['spy_numHomeWinsRatio_H2H'] = np.where(submit_df['Team2_spy_homeWins_H2H'] != 0, 
                                            submit_df['Team1_spy_homeWins_H2H'] / submit_df['Team2_spy_homeWins_H2H'], 0)
submit_df['spy_WinRatioRatio_H2H'] = np.where(submit_df['Team2_spy_winRatio_H2H'] != 0, 
                                            submit_df['Team1_spy_winRatio_H2H'] / submit_df['Team2_spy_winRatio_H2H'], 0)
submit_df['spy_homeWinRatioRatio_H2H'] = np.where(submit_df['Team2_spy_homeWinRatio_H2H'] != 0, 
                                            submit_df['Team1_spy_homeWinRatio_H2H'] / submit_df['Team2_spy_homeWinRatio_H2H'], 0)

In [ ]:
submit_df.shape

In [ ]:
submit_df.head()

In [ ]:
xgb_model = pickle_load('/kaggle/working/xgb_model.pkl')
# modelCols = [col for col in submit_df.columns if 'spy_' in col]
predictions = xgb_model.predict_proba(submit_df[keepCols])

### Create submission file

In [ ]:
submission = submit_df[['ID']].copy()
submission['Pred'] = predictions[:,1]
submission.head()

In [ ]:
sns.displot(submission['Pred'],bins=100)

In [ ]:
submission.to_csv('/kaggle/working/submission.csv', index=False)

In [ ]:
# Bring in detailed results dataframe
# Head to head features

In [ ]:
# Loss records for existing themes
# Venue based win/loss record
# Coach win/loss record, tenure
# Season/ NCAA based win/loss record
# Points, points difference for win/loss for all themes

# Average win/loss per season
# Average points per season

# Team head to head record (win/loss, points, venue based)
# Coach head to head record (win/loss, points, venue based)
# Sead based head to head record (win/loss, points, venue based)

# OT based stats
# Most recent games (form, recency, days per game)
# How many years in NCAA, how many seasons played, how many times champions, what position finish on avg